In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io
import seaborn as sns
from scipy.stats import norm
from scipy.stats import power
from scipy.stats import ttest_1samp
from scipy.stats import ttest_rel
from scipy.stats import ttest_ind
from scipy.stats import ttest_ind_from_stats
import statsmodels.api as sm
from statsmodels.stats.proportion import  proportions_ztest
from scipy.stats import chisquare # Statistical test (chistat, pvalue)
from scipy.stats import chi2_contingency
from scipy.stats import chi2
from scipy.stats import f_oneway
from statsmodels.graphics.gofplots import qqplot
from statsmodels.formula.api import ols
from scipy.stats import kruskal
from scipy.stats import shapiro
from scipy.stats import levene
from scipy.stats import kstest
from scipy.stats import ks_2samp

from scipy.stats import pearsonr, spearmanr

### Q1. Store Satisfaction Analysis
A Data Analyst at a retail company wants to analyze if there is any significant difference in average customer satisfaction scores (out of 10) across three store locations. The sample data is:

Store A → [8.2, 8.5, 7.9, 8.1, 8.3, 8.6]  
Store B → [7.4, 7.8, 7.5, 7.3, 7.6, 7.7]  
Store C → [8.8, 8.7, 9.0, 8.9, 8.6, 8.5]


The analyst:
<ol>
<li>Checks for normality using the Shapiro-Wilk test.</li>
<li>Uses Levene’s test to check for equality of variances.</li>
<li>Performs One-Way ANOVA if assumptions are met, else performs Kruskal-Wallis Test.</li>
<li>If ANOVA is significant, perform pairwise t-tests to identify the specific pairs that differ.</li>
<li>Significance level: 0.05</li>
</ol>
What is the correct conclusion?

In [15]:
storeA = [8.2, 8.5, 7.9, 8.1, 8.3, 8.6]  
storeB = [7.4, 7.8, 7.5, 7.3, 7.6, 7.7]  
storeC = [8.8, 8.7, 9.0, 8.9, 8.6, 8.5]
alpha = 0.05

s_stat, p_value = shapiro(storeA)
print(f'Store A -> s_stat {s_stat}, p_value: {p_value:.6f}')
print(f'({'Normal distribution' if p_value > 0.05 else 'Not normal distribution'})')
s_stat, p_value = shapiro(storeB)
print(f'Store B -> s_stat {s_stat}, p_value: {p_value:.6f}')
print(f'({'Normal distribution' if p_value > 0.05 else 'Not normal distribution'})')
s_stat, p_value = shapiro(storeC)
print(f'Store C -> s_stat {s_stat}, p_value: {p_value:.6f}')
print(f'({'Normal distribution' if p_value > 0.05 else 'Not normal distribution'})')

l_stat, p_value = levene(storeA, storeB, storeC)
print(f'Levene -> s_stat {l_stat}, p_value: {p_value:.10f}')
print(f'({'Equal Variances' if p_value > 0.05 else 'Unequal Variances'})')

f_stat, p_value = f_oneway(storeA, storeB, storeC)
print(f'One Way -> s_stat {f_stat}, p_value: {p_value:.10f}')
print(f'({'similar satisfaction' if p_value > 0.05 else 'different satisfaction'})')

kw_stat, p_value = kruskal(storeA, storeB, storeC)
print(f'KruskaWallis -> s_stat {kw_stat}, p_value: {p_value:.10f}')
print(f'({'similar satisfaction' if p_value > 0.05 else 'different satisfaction'})')


Store A -> s_stat 0.9787743974156957, p_value: 0.945306
(Normal distribution)
Store B -> s_stat 0.9818894288744925, p_value: 0.960555
(Normal distribution)
Store C -> s_stat 0.9818894288744917, p_value: 0.960555
(Normal distribution)
Levene -> s_stat 0.42857142857143155, p_value: 0.6591699522
(Equal Variances)
One Way -> s_stat 48.012195121951265, p_value: 0.0000003020
(different satisfaction)
KruskaWallis -> s_stat 14.39227852464667, p_value: 0.0007494738
(different satisfaction)


### Q2. Store & Channel Sales
A data analyst at a retail company is analyzing the impact of two factors—Store Location (Urban, Suburban, Rural) and Sales Channel (Online, In-store)—on monthly product sales.<br/>
The goal is to test for:<br/>

<ul>
<li>Main effect of Store Location</li>
<li>Main effect of Sales Channel</li>
<li>Interaction effect between the two factors</li>
</ul><br/>
Monthly sales (USD):
<ul>
<li>Urban, Online     → [450, 470, 460, 455, 480]  </li>
<li>Urban, In-store   → [550, 560, 530, 540, 570]  </li>
<li>Suburban, Online  → [420, 430, 410, 440, 450]  </li>
<li>Suburban, In-store→ [490, 505, 480, 495, 510]  </li>
<li>Rural, Online     → [350, 370, 360, 340, 380]  </li>
<li>Rural, In-store   → [460, 470, 490, 480, 460]  </li>
</ul>
Significance level: alpha = 0.05

Which of the following interpretations is statistically valid?


In [20]:
# 1. Prepare the dataset
data = {
    'Location': ['Urban']*10 + ['Suburban']*10 + ['Rural']*10,
    'Channel': (['Online']*5 + ['In-store']*5) * 3,
    'Sales': [
        450, 470, 460, 455, 480,  # Urban Online
        550, 560, 530, 540, 570,  # Urban In-store
        420, 430, 410, 440, 450,  # Suburban Online
        490, 505, 480, 495, 510,  # Suburban In-store
        350, 370, 360, 340, 380,  # Rural Online
        460, 470, 490, 480, 460   # Rural In-store
    ]
}

df = pd.DataFrame(data)
df

,Location,Channel,Sales
0,Urban,Online,450
1,Urban,Online,470
2,Urban,Online,460
3,Urban,Online,455
4,Urban,Online,480
5,Urban,In-store,550
6,Urban,In-store,560
7,Urban,In-store,530
8,Urban,In-store,540
9,Urban,In-store,570


In [23]:
# TOW-WAY ANOVA
model = ols('Sales ~ C(Location)*C(Channel)',data=df).fit()
anova_table = sm.stats.anova_lm(model,typ=2)

print(f'results of anova_table:\n{anova_table}')

alpha = 0.05
for factor in ['C(Location)', 'C(Channel)', 'C(Location):C(Channel)']:
    p_val = anova_table.loc[factor, 'PR(>F)']
    status = "Significant" if p_val < alpha else "Not Significant"
    print(f"{factor}: p-value = {p_val:.5f} ({status})")


results of anova_table:
                              sum_sq    df           F        PR(>F)
C(Location)             40971.666667   2.0  101.792961  1.891430e-12
C(Channel)              58520.833333   1.0  290.786749  6.449137e-15
C(Location):C(Channel)   2651.666667   2.0    6.587992  5.240652e-03
Residual                 4830.000000  24.0         NaN           NaN
C(Location): p-value = 0.00000 (Significant)
C(Channel): p-value = 0.00000 (Significant)
C(Location):C(Channel): p-value = 0.00524 (Significant)


### Q3. Income Distribution Test

A Data Scientist at a financial institution wants to compare the distribution of monthly income (in thousands of dollars) between the Sales and Marketing departments. The goal is to check if the two departments have significantly different income distributions.

Income Data:

Sales Department     → [38, 45, 42, 40, 35, 50, 41, 43, 47, 46]  
Marketing Department → [50, 55, 60, 52, 57, 54, 61, 59, 63, 62]

The Data Scientist wants to test if the monthly income distributions of the two departments are different or not. The significance level (α) is set to 0.05.

What is the correct conclusion based on the results of the KS test for comparing the distributions of the two groups?

In [29]:
sales = [38, 45, 42, 40, 35, 50, 41, 43, 47, 46]  
marketing =  [50, 55, 60, 52, 57, 54, 61, 59, 63, 62]
alpha = 0.05

ks_stat, p_value = kstest(sales,marketing)
print(f'ks_stat: {ks_stat:.9f} and p_value: {p_value:.9f}')

print(f'({'Same distributions' if p_value > alpha else 'Significant different distributions'})')

ks_stat: 0.900000000 and p_value: 0.000216502
(Significant different distributions)


### Q4. Stock Return Correlation
A financial analyst is examining the relationship between the daily returns (in %) of two stocks over the past 10 days:

Stock A Returns → [1.2, 1.5, 1.7, 1.6, 1.8, 1.5, 1.6, 1.7, 1.4, 1.5]  
Stock B Returns → [0.9, 1.0, 1.2, 1.3, 1.4, 1.2, 1.3, 1.4, 1.1, 1.2]

The analyst wants to determine the relationship between the returns of Stock A and Stock B and assess whether the returns of the two stocks are positively correlated, negatively correlated, or uncorrelated.

Which of the following conclusions can the analyst draw based on the covariance and correlation between the two stocks' returns?

In [34]:
data = pd.DataFrame({'StockA':[1.2, 1.5, 1.7, 1.6, 1.8, 1.5, 1.6, 1.7, 1.4, 1.5],
                     'StockB':[0.9, 1.0, 1.2, 1.3, 1.4, 1.2, 1.3, 1.4, 1.1, 1.2]})

print(f'corelation data is: \n{data.corr()}')

pearsonr(data['StockA'], data['StockB'])


corelation data is: 
          StockA    StockB
StockA  1.000000  0.872357
StockB  0.872357  1.000000


PearsonRResult(statistic=np.float64(0.8723567442899586), pvalue=np.float64(0.0009927667987489372))

In [36]:
df_ride = pd.read_csv('../../assets/Ride_Sharing_Service.csv')
df_ride.head()

,ride_id,user_id,ride_type,distance_km,duration_min,fare_amount,tip_amount,pickup_area,dropoff_area,user_rating,payment_method,user_type,surge_multiplier,feedback
0,RIDE1000,USER2000,Shared,6.85,58.72,257.55,10.0,Suburb,Suburb,2,UPI,New,1.89,NaN
1,RIDE1001,USER2001,NaN,12.99,15.80,113.77,0.0,Suburb,Industrial,4,NaN,NaN,1.53,NaN
2,RIDE1002,USER2002,Standard,22.80,21.01,141.85,0.0,Suburb,Suburb,2,UPI,New,2.06,Good
3,RIDE1003,USER2003,Shared,11.97,24.39,257.99,15.0,Airport,Uptown,2,NaN,New,1.72,Good
4,RIDE1004,USER2004,Shared,10.61,10.81,139.86,10.0,Airport,Airport,3,Card,NaN,1.57,NaN


### Q8. Customer Segmentation by Distance


In [45]:
bins = [0.0, 5.0, 15.0, np.inf]
names = ['Short', 'Medium', 'Long']
df_ride['dist_type'] = pd.cut(df_ride['distance_km'], bins=bins, labels=names, include_lowest=True)
df_ride['dist_type'].value_counts()

dist_type
Medium    35
Short      8
Long       7
Name: count, dtype: int64

### Q9. User Type vs Payment Preference

In [56]:
from sklearn.preprocessing import LabelEncoder
df_ride['user_type'] = df_ride['user_type'].fillna('Unknown')
df_ride['user_type_encoded'] = LabelEncoder().fit_transform(df_ride['user_type'])

trends = df_ride.groupby(['user_type', 'payment_method']).size().reset_index(name='count')
trends

,user_type,payment_method,count
0,Loyal,Card,2
1,Loyal,Cash,4
2,Loyal,UPI,1
3,Loyal,Wallet,4
4,New,Card,5
5,New,Cash,3
6,New,UPI,3
7,New,Wallet,5
8,Unknown,Card,2
9,Unknown,Cash,3


### Q10. Normalizing Features for Clustering

In [59]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# Normalize selected features
df_ride[['distance_km_norm', 'fare_amount_norm']] = scaler.fit_transform(df_ride[['distance_km', 'fare_amount']])

# Preview results
df_ride[['distance_km', 'distance_km_norm', 'fare_amount', 'fare_amount_norm']].head()

# Answer: Min-Max normalization scales values to [0, 1], which keeps distances meaningful and bounded — ideal for clustering.

,distance_km,distance_km_norm,fare_amount,fare_amount_norm
0,6.85,0.218707,257.55,0.554718
1,12.99,0.500229,113.77,0.072946
2,22.80,0.950023,141.85,0.167035
3,11.97,0.453462,257.99,0.556192
4,10.61,0.391105,139.86,0.160367


### Q11. Removing Fare Outliers for Area-Wise Analysis

In [60]:
# 1. Calculate Q1 (25th percentile) and Q3 (75th percentile)
Q1 = df_ride['fare_amount'].quantile(0.25)
Q3 = df_ride['fare_amount'].quantile(0.75)
IQR = Q3 - Q1

# 2. Define bounds for filtering
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 3. Create the filtered DataFrame
df_cleaned = df_ride[(df_ride['fare_amount'] >= lower_bound) & (df_ride['fare_amount'] <= upper_bound)]

# 4. Calculate the groupwise mean by pickup_area
average_fares = df_cleaned.groupby('pickup_area')['fare_amount'].mean().reset_index()

print(average_fares)

  pickup_area  fare_amount
0     Airport   230.368889
1  Industrial   196.504615
2      Suburb   205.355385
3      Uptown   194.621667


### Q12. Feature Engineering for Customer Generosity

In [61]:
df_ride['tip_percent'] = (df_ride['tip_amount']/df_ride['fare_amount'])*100
df_ride['is_generous'] = (df_ride['tip_percent'] > 5).astype(int)
print('count of generous rides:', df_ride[df_ride['is_generous'] == 1].shape[0])

generous_ride_types = df_ride.groupby('ride_type')['is_generous'].mean().sort_values(ascending=False)
print(generous_ride_types)
df_ride[df_ride['is_generous'] == 1].value_counts('ride_type')

count of generous rides: 12
ride_type
Shared      0.461538
Standard    0.200000
Premium     0.000000
Name: is_generous, dtype: float64


ride_type
Shared      6
Standard    2
Name: count, dtype: int64

### Q14. Tip Behavior by Ride Type Considering Surge Multiplier

In [64]:
df_ride['tip_by_distance'] = df_ride['tip_amount']/df_ride['distance_km']
df_ride.groupby('ride_type')['tip_by_distance'].mean().sort_values(ascending=False)

# Answer: Riders in Shared and Premium categories tip more per km than those in Standard rides, suggesting perceived ride value impacts tipping behavior.

ride_type
Shared      0.924264
Premium     0.845513
Standard    0.765280
Name: tip_by_distance, dtype: float64